# Advanced Retrieval with Self-Querying and Structured Metadata Filtering

Welcome to the advanced section of our RAG curriculum. While basic retrieval often relies solely on semantic similarity (i.e., finding documents whose embeddings are closest to the query embedding), real-world applications rarely operate in a vacuum. Users frequently need to narrow down results based on explicit constraints—for example, "Show me sci-fi movies directed by Christopher Nolan *after* 2010." Simply relying on vector similarity might pull up relevant but incorrect documents (e.g., pulling up an action movie when the user meant a drama).

This notebook introduces **Self-Querying**, a critical pattern for building robust, production-grade RAG systems. Self-querying involves using a Large Language Model (LLM) to analyze the incoming natural language query and *translate* it into structured parameters—such as field names, values, and comparison operators (`eq`, `gt`, `lt`)—before the retrieval step even executes. Instead of passing the raw text to the retriever, we pass a structured filter object. This dramatically increases the precision and reliability of the retrieved context, ensuring that the LLM receives only the most relevant subset of documents.

Understanding how to implement this pattern is essential for advanced development using frameworks like LangGraph. In a complex agentic workflow, the self-query step acts as a crucial decision node: it determines *how* the retriever should behave (e.g., "Do I need to filter by year?" or "Is simple semantic search enough?"). By mastering structured output parsing and metadata filtering, you move beyond basic question-answering and build sophisticated knowledge agents capable of handling complex, multi-faceted user intent.

### Learning Objectives

Upon completing this notebook, you will be able to:

*   **Understand the limitations of pure semantic retrieval:** Recognize scenarios where simple vector search fails due to lack of explicit constraints.
*   **Implement Structured Output Parsing:** Utilize Pydantic models and LLMs to reliably extract structured metadata (like field names, values, and operators) from natural language queries.
*   **Apply Advanced Filtering in RAG:** Integrate the parsed structure into a vector store's native filtering mechanism (e.g., Chroma's `filter` parameter).
*   **Design Self-Querying Agents:** Conceptualize how an LLM can act as a pre-processor, transforming ambiguous user intent into precise retrieval instructions for a complex RAG pipeline.


### Setup and Imports

This cell imports all necessary libraries and components for building a sophisticated RAG pipeline. It includes utilities for environment variable management (`dotenv`), type hinting, core LangChain components (like `Document` and `BaseRetriever`), prompt templating, vector store interaction (`Chroma`), and embedding/LLM models (`OpenAIEmbeddings`, `ChatOpenAI`).


In [13]:
from dotenv import load_dotenv # Used to load environment variables from a .env file
from typing import Any, Optional # Standard Python type hinting for robust code
from langchain_core.documents import Document # Core class representing a document chunk (text + metadata)
from langchain_core.retrievers import BaseRetriever # Abstract base class for all retrievers in LangChain
from langchain_core.prompts import ChatPromptTemplate # Used to structure prompts for chat models
from langchain_chroma import Chroma # Specific implementation of a vector store (ChromaDB)
from langchain_openai import OpenAIEmbeddings, ChatOpenAI # Components for generating embeddings and interacting with OpenAI's chat model
from pydantic import BaseModel, Field # Used for defining structured output schemas using Pydantic


In [14]:
load_dotenv()

True

### Initialization of Core Components

This cell initializes the essential components for our RAG system: an embedding model and a Large Language Model (LLM). `OpenAIEmbeddings` converts text into numerical vectors, while `ChatOpenAI` provides the generative AI capabilities needed for reasoning and response generation.


In [20]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small") # Initializes the embedding model used to convert text chunks into vector representations.
llm = ChatOpenAI(model="gpt-5", temperature=0) # Initializes the chat LLM, which will handle reasoning and generating final answers. Setting temperature=0 ensures deterministic, reliable outputs.


### Document Loading and Setup

This cell initializes the knowledge base by creating a list of `Document` objects. Each document represents movie information (plot summary, metadata) and is crucial for providing context to the subsequent RAG steps.


In [16]:
docs = [
    Document(
        page_content="A masked vigilante fights crime in a corrupt city with the help of a billionaire's technology. An iconic supervillain pushes him to his limits in a battle for Gotham's soul.",
        metadata={"title": "The Dark Knight", "genre": "action", "year": 2008, "rating": 9.0, "director": "Christopher Nolan"},
    ),
    Document(
        page_content="A thief who steals secrets through dream-sharing technology is offered a chance to have his past erased if he can plant an idea in someone's mind. A visually stunning exploration of the subconscious.",
        metadata={"title": "Inception", "genre": "sci-fi", "year": 2010, "rating": 8.8, "director": "Christopher Nolan"},
    ),
    Document(
        page_content="A team of explorers travels through a wormhole in space to find a new habitable planet for humanity. Stunning visuals of black holes and time dilation challenge our understanding of physics.",
        metadata={"title": "Interstellar", "genre": "sci-fi", "year": 2014, "rating": 8.6, "director": "Christopher Nolan"},
    ),
    Document(
        page_content="A programmer discovers that reality is a simulation and joins a rebellion against the machines controlling humanity. A groundbreaking blend of philosophy, martial arts, and bullet-time action.",
        metadata={"title": "The Matrix", "genre": "sci-fi", "year": 1999, "rating": 8.7, "director": "Lana Wachowski"},
    ),
    Document(
        page_content="Two criminals and a mob boss's wife are caught in a web of violence and dark humor over a single eventful day in Los Angeles. Interweaving storylines told out of chronological order.",
        metadata={"title": "Pulp Fiction", "genre": "drama", "year": 1994, "rating": 8.9, "director": "Quentin Tarantino"},
    ),
    Document(
        page_content="A maverick surgeon navigates the chaotic social landscape of a mobile army unit during the Korean War. Sharp satirical comedy disguised as a war film, later adapted into a beloved TV series.",
        metadata={"title": "MASH", "genre": "comedy", "year": 1970, "rating": 7.4, "director": "Robert Altman"},
    ),
    Document(
        page_content="Humanity sends a last-ditch mission to reignite the dying sun with a massive stellar bomb. An intense psychological thriller set in the terrifying emptiness of deep space.",
        metadata={"title": "Sunshine", "genre": "sci-fi", "year": 2007, "rating": 7.3, "director": "Danny Boyle"},
    ),
    Document(
        page_content="A soldier wakes up in another man's body aboard a commuter train just minutes before it explodes, reliving the event repeatedly to identify the bomber. A clever sci-fi thriller about time loops and identity.",
        metadata={"title": "Source Code", "genre": "sci-fi", "year": 2011, "rating": 7.5, "director": "Duncan Jones"},
    ),
]

print(f"Created {len(docs)} movie documents")


Created 8 movie documents


### Vector Store Initialization (Chroma)

This cell initializes a persistent vector store using ChromaDB. It takes the loaded documents (`docs`) and embeds them using the specified `embeddings` model, storing the resulting vectors in a collection named `movies_collection`. This step is crucial for enabling efficient semantic search over your document corpus.


In [40]:
vectorstore = Chroma.from_documents(docs, 
                                    embedding=embeddings, 
                                    collection_name="movies_collection")


query --> llm --> Pydantic Schema --> Structured Output (Pydantic Object) --> Translate

1. Semantic Part
2. Metadata part


[filter(field=year, value=2005, operator="gte") year >= 2005,
filter(filed=title, value=inception, operator=eq)]

### Schema Definition for Structured Output

These Pydantic models define the structured output format expected from an LLM when processing a user's natural language query. `MetadataFilter` captures individual filtering conditions (field, value, operator), and `SelfQuerySchema` uses these filters to structure both the core semantic search query and any associated metadata constraints.


In [41]:
# MetadataFilter captures a single filter condition with an optional comparison operator

# Defines the structure for a single metadata filter.
class MetadataFilter(BaseModel):
    
    field: str = Field(description="The metadata field name to filter on") # The specific field in the document metadata.
    value: str | int | float = Field(description="The value to filter by") # The target value for the filter.
    operator: str = Field(default="eq", description="Comparison operator: eq, ne, gt, gte, lt, lte") # Comparison type (e.g., equals, not equal).
    

# SelfQuerySchema is the structured output the LLM produces from a natural language query
# This model dictates how the LLM should decompose a complex user query.
class SelfQuerySchema(BaseModel):
    query: str = Field(description="The semantic search query extracted from the user's query") # The core text query for vector search.
    filters: Optional[list[MetadataFilter]] = Field(
        default=None,
        description="Metadata filters extracted from the query. None if no filters apply."
    )



This cell sets up a structured query parser using an LLM. It defines a system prompt that instructs the model to extract both a semantic search query and specific metadata filters (like genre or year) from natural language input, ensuring the output conforms strictly to the `SelfQuerySchema` Pydantic structure.


In [42]:
# System prompt tells the LLM what fields are available and how to populate the schema
system_prompt = """You are a query parser for a movie database. Parse the user's natural language query into:
1. A semantic search query (the conceptual meaning to search for)
2. Optional metadata filters on fields: title (string), genre (string), year (integer), rating (float), director (string)

Supported operators: eq, ne, gt, gte, lt, lte
Only add filters when the user explicitly specifies metadata constraints."""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "query: {query}"),
])

# with_structured_output binds the Pydantic schema to the LLM — output is always a SelfQuerySchema instance
structured_llm = llm.with_structured_output(SelfQuerySchema)

query_chain = prompt | structured_llm   # pydantic object as output


### Code Explanation

This cell executes the `query_chain` with a specific natural language query. By accessing `.filters` on the result, we extract structured metadata (like director or release year) that the underlying LLM/retrieval system determined was relevant to answering the question, making these filters available for subsequent steps.


In [31]:
filters = query_chain.invoke({"query": "What are some movies directed by Christopher Nolan released after 2005"}).filters


This cell is likely used to display or inspect the available filter options (e.g., metadata filters) that can be applied when querying a vector store or knowledge base. It helps the user understand what filtering criteria are possible for advanced retrieval.


In [32]:
filters


[MetadataFilter(field='director', value='Christopher Nolan', operator='eq'),
 MetadataFilter(field='year', value=2005, operator='gt')]

### Code Explanation

This cell executes the `query_chain` (likely a LangChain Expression Language or LCEL chain) with a specific query string. It simulates running the advanced RAG pipeline to retrieve and synthesize an answer based on the provided question, demonstrating the final invocation step.


In [35]:
query_chain.invoke({"query": "What are some movies directed by Christopher Nolan released after 2005"}) # Invokes the pre-configured query chain (e.g., a LangGraph or LCEL pipeline) with the specific user query to execute the RAG process and retrieve an answer.


SelfQuerySchema(query='movies directed by Christopher Nolan released after 2005', filters=[MetadataFilter(field='director', value='Christopher Nolan', operator='eq'), MetadataFilter(field='year', value=2005, operator='gt')])

### Custom Self-Query Retriever Implementation

This class implements a specialized retriever that enhances standard vector search by first using an LLM to parse the natural language query into structured metadata filters (e.g., genre, year). It then translates these structured filters into the specific format required by ChromaDB and executes a filtered similarity search.


In [43]:
class CustomSelfQueryRetriever(BaseRetriever):
    """Retriever that uses an LLM to parse natural language queries into structured filters."""

    vectorstore: Any  # Chroma instance
    query_chain: Any  # prompt | structured_llm chain

    def _build_chroma_filter(self, filters: list[MetadataFilter]) -> dict:
        # Chroma supports operator-based filters: {"field": {"$op": value}}
        # For equality ("eq") the shorthand "field": value} also works, but we use the full form for consistency
        op_map = {"eq": "$eq", "ne": "$ne", "gt": "$gt", "gte": "$gte", "lt": "$lt", "lte": "$lte"}
        if len(filters) == 1:
            f = filters[0]
            # If only one filter, return a simple dictionary structure
            return {f.field: {op_map.get(f.operator, "$eq"): f.value}}  # {'director': {"$eq": "Christopher Nolan"}}
        # Multiple filters are combined with $and for logical AND operation
        return {"$and": [{f.field: {op_map.get(f.operator, "$eq"): f.value}} for f in filters]} # {"$and": [{"genre": {"$eq": "sci-fi"}}, {"year": {"$gt": 2005}}]}

    def _get_relevant_documents(self, query: str) -> list[Document]:
        # Step 1: parse the natural language query into a structured schema using the LLM chain
        parsed: SelfQuerySchema = self.query_chain.invoke({"query": query})
        print(f"Parsed query: '{parsed.query}'  |  Filters: {parsed.filters}")

        # Step 2: build Chroma-compatible filter if any filters were extracted
        chroma_filter = None
        if parsed.filters:
            # Use the helper method to convert structured filters into a dictionary format for ChromaDB
            chroma_filter = self._build_chroma_filter(parsed.filters)

        # Step 3: run similarity search with the semantic query and optional metadata filter
        # The vectorstore handles both the embedding search (query) and the filtering (filter=...) 
        return self.vectorstore.similarity_search(parsed.query, k=3, filter=chroma_filter)


### Custom Self-Query Retriever Initialization

The `CustomSelfQueryRetriever` is initialized here. This specialized retriever automatically generates optimal queries for the vector store by leveraging a separate query chain, making it highly effective for complex or ambiguous user inputs.


In [37]:
retriever = CustomSelfQueryRetriever(vectorstore=vectorstore, query_chain=query_chain)


### Code Explanation

This line is crucial for preparing the retriever to use advanced filtering capabilities. It calls a method on the `retriever` object, passing in the defined `filters`, which instructs the underlying Chroma vector store how to apply metadata constraints during retrieval.


In [34]:
retriever._build_chroma_filter(filters)


{'$and': [{'director': {'$eq': 'Christopher Nolan'}}, {'year': {'$gt': 2005}}]}

### Code Explanation

This cell initializes and uses a `CustomSelfQueryRetriever`. This specialized retriever is crucial because it doesn't just perform a standard vector search; instead, it leverages an internal query generation chain (`query_chain`) to intelligently rewrite the user's natural language query (e.g., adding filters like 'on and after 2005') before querying the vector store, thereby improving retrieval accuracy for complex, filtered requests.


In [44]:
retriever = CustomSelfQueryRetriever(vectorstore=vectorstore, query_chain=query_chain)

# Query 1: genre + year filter
print("=== Sci-fi movies on and after 2005 ===")

results = retriever.invoke("What are some sci-fi movies released on and after 2005?")
for doc in results:
    print(f"  [{doc.metadata['year']}] {doc.metadata['title']} ({doc.metadata['genre']})")
print()


=== Sci-fi movies on and after 2005 ===
Parsed query: 'sci-fi movies'  |  Filters: [MetadataFilter(field='genre', value='sci-fi', operator='eq'), MetadataFilter(field='year', value=2005, operator='gte')]
  [2007] Sunshine (sci-fi)
  [2014] Interstellar (sci-fi)
  [2011] Source Code (sci-fi)



### Code Explanation

This cell demonstrates a pure semantic retrieval query, meaning the search relies solely on the textual content of the prompt without applying any metadata filters. It uses the `retriever` object (likely an instance of a vector store retriever) to fetch documents relevant to 'Movies about dreams and the subconscious mind' and then prints key details (year, title, genre) from the retrieved document metadata.


In [45]:
# Query 2: pure semantic — no metadata filter expected

print("=== Movies about dreams ===")
results = retriever.invoke("Movies about dreams and the subconscious mind")
for doc in results:
    print(f"  [{doc.metadata['year']}] {doc.metadata['title']} ({doc.metadata['genre']})")
print()


=== Movies about dreams ===
Parsed query: 'movies about dreams and the subconscious mind; dreamscapes, surreal, subconscious, lucid dreaming, mind-bending narratives'  |  Filters: None
  [2010] Inception (sci-fi)
  [2014] Interstellar (sci-fi)
  [2011] Source Code (sci-fi)



### Query Execution and Filtering

This cell executes a specific query using the `retriever` component. It demonstrates how to retrieve relevant documents (movies directed by Christopher Nolan) and then iterates through the results, printing key metadata like year, title, and rating for easy consumption.


In [46]:
# Query 3: director equality filter
print("=== Christopher Nolan movies ===")
# Invoke the retriever with a specific query string.
# The retriever handles the search logic against the knowledge base.
results = retriever.invoke("What movies did Christopher Nolan direct?")
# Iterate through all retrieved documents (Document objects).
for doc in results:
    # Access and print key metadata fields from each document for structured output.
    print(f"  [{doc.metadata['year']}] {doc.metadata['title']} - Rating: {doc.metadata['rating']}")


=== Christopher Nolan movies ===
Parsed query: 'movies directed by Christopher Nolan'  |  Filters: [MetadataFilter(field='director', value='Christopher Nolan', operator='eq')]
  [2008] The Dark Knight - Rating: 9.0
  [2010] Inception - Rating: 8.8
  [2014] Interstellar - Rating: 8.6
